In [ ]:
import pyspark.sql as sql
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark : sql.SparkSession = (
	sql.SparkSession.builder
	.appName("etl_station_to_station")
	.getOrCreate()
)

NB_DECIMALS = 6

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/04 13:04:43 WARN Utils: Your hostname, BEBEL-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/04 13:04:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/04 13:04:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/04 13:04:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df : sql.DataFrame = spark.read.csv("data/station_to_station.csv", header=True, sep=";", inferSchema=True)
df.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
df.printSchema()

root
 |-- Geo Shape: string (nullable = true)
 |-- Gare de départ (id): integer (nullable = true)
 |-- Gare de départ: string (nullable = true)
 |-- Gare d'arrivée (id): integer (nullable = true)
 |-- Gare d'arrivée: string (nullable = true)
 |-- Distance: double (nullable = true)
 |-- geo_point_2d: string (nullable = true)



# Data validation

## Null values

In [ ]:
null_counts = []
for column in df.columns:
	null_count = df.filter(F.col(column).isNull()).count()
	null_counts.append((column, null_count))
for col_name, null_count in null_counts:
	print(f"Column '{col_name}' has {null_count} null values")

Column 'Geo Shape' has 0 null values
Column 'Gare de départ (id)' has 0 null values
Column 'Gare de départ' has 0 null values
Column 'Gare d'arrivée (id)' has 0 null values
Column 'Gare d'arrivée' has 0 null values
Column 'Distance' has 0 null values
Column 'geo_point_2d' has 0 null values


## Unique values

In [ ]:
unique_counts = []
for column in df.columns :
	unique_count : int = df.agg(F.count_distinct(column).alias('count')).collect()[0]['count']
	unique_counts.append((column, unique_count))
for col_name, unique_count in unique_counts :
	print(f"Column {col_name} has {unique_count} unique values.")

Column Geo Shape has 708 unique values.
Column Gare de départ (id) has 559 unique values.
Column Gare de départ has 559 unique values.
Column Gare d'arrivée (id) has 559 unique values.
Column Gare d'arrivée has 559 unique values.
Column Distance has 733 unique values.
Column geo_point_2d has 708 unique values.


## Empty values

In [ ]:
empty_counts = []
string_columns = [f.name for f in df.schema.fields if isinstance(f.dataType, T.StringType)]
for column in string_columns:
	empty_count = df.filter(F.col(column) == "").count()
	empty_counts.append((column, empty_count))
for col_name, empty_count in empty_counts:
	print(f"Column '{col_name}' has {empty_count} empty values")

Column 'Geo Shape' has 0 empty values
Column 'Gare de départ' has 0 empty values
Column 'Gare d'arrivée' has 0 empty values
Column 'geo_point_2d' has 0 empty values


## Duplicate values

In [ ]:
duplicates = df.groupBy(df.columns).count().filter(F.col("count") > 1)
duplicate_count = duplicates.count()
if duplicate_count > 0:
	print(f"There are {duplicate_count} duplicate lines in the data")
else:
	print("No duplicate lines found in the data")

No duplicate lines found in the data


# Data transformation

## Converting and Renaming the column "Geo Shape"

In [ ]:
cleaned_coordinates : sql.Column = F.regexp_replace(F.col("Geo Shape"), r'(^")|("$)', "")
cleaned_coordinates = F.regexp_replace(cleaned_coordinates, r'""', '"')

def inverse_and_truncate(coordinates : list[list[float]]) -> list[list[float]] :
	new_coordinates : list[list[float]] = []
	for coord in coordinates :
		new_coordinates.append([round(coord[1], NB_DECIMALS), round(coord[0], NB_DECIMALS)])
	return new_coordinates

inverse_and_truncate_udf = F.udf(inverse_and_truncate, T.ArrayType(T.ArrayType(T.DoubleType())))

schema : T.StructType = T.StructType([
	T.StructField("coordinates", T.ArrayType(T.ArrayType(T.DoubleType())), True),
	T.StructField("type", T.StringType(), True),
])

parsed : sql.DataFrame = df.withColumn("parsed_geo_shape", F.from_json(cleaned_coordinates, schema))

new_df : sql.DataFrame = parsed.withColumn(
	"Coordinates", 
	F.to_json(inverse_and_truncate_udf(F.col("parsed_geo_shape.coordinates")))
).drop("parsed_geo_shape", "Geo Shape", "parsed_geo_shape")

new_df.show(5, truncate=False)

+-------------------+--------------+-------------------+--------------+------------------+--------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Renaming the column "Gare de départ (id)"

In [9]:
new_df = new_df.withColumnRenamed("Gare de départ (id)", "Departure_station_id")
new_df.show(5, truncate=False)

+--------------------+--------------+-------------------+--------------+------------------+--------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Renaming the column "Gare d'arrivée (id)"

In [10]:
new_df = new_df.withColumnRenamed("Gare d'arrivée (id)", "Arrival_station_id")
new_df.show(5, truncate=False)

+--------------------+--------------+------------------+--------------+------------------+--------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Dropping the station name columns

In [11]:
new_df = new_df.drop("Gare de départ", "Gare d'arrivée")
new_df.show(5, truncate=False)

+--------------------+------------------+------------------+--------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Converting and renaming the column "geo_point_2d"

In [ ]:
def truncate(value : float) -> float :
	if value is None :
		return None
	return round(value, NB_DECIMALS)

truncate_udf = F.udf(truncate, T.DoubleType())

new_df = new_df.withColumn(
	"Geo_x",
	truncate_udf(F.split(F.col("geo_point_2d"), ",")[0].cast("double")),
).withColumn(
	"Geo_y",
	truncate_udf(F.split(F.col("geo_point_2d"), ",")[1].cast("double"))
).drop("geo_point_2d")
new_df.show(5, truncate=False)

+--------------------+------------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Truncate the values in the column "Distance"

In [ ]:
NB_DECIMALS = 2
truncate_udf = F.udf(truncate, T.DoubleType())
new_df = new_df.withColumn(
	"Distance",
	truncate_udf(F.col("Distance")),
)
new_df.show(5, truncate=False)

+--------------------+------------------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Reordering the columns

In [ ]:
final_df : sql.DataFrame = new_df.select(
	"Departure_station_id",
	"Arrival_station_id",
	"Geo_x",
	"Geo_y",
	"Distance",
	"Coordinates"
)
final_df.show(5, truncate=False)

+--------------------+------------------+---------+--------+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
final_df.printSchema()

root
 |-- Departure_station_id: integer (nullable = true)
 |-- Arrival_station_id: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Coordinates: string (nullable = true)



# Data loading

In [16]:
final_df.coalesce(1).write.mode("overwrite").option("header", True).csv("cleaned_data/station_to_station", sep=";")